# Experiment: Qwen3-8B BNB NF4 typed full LLM multi-agent TEC evaluation

This notebook evaluates the existing typed full LLM multi-agent TEC
architecture using the text-only Qwen/Qwen3-8B model loaded through
Transformers with dynamic BitsAndBytes NF4 4-bit quantization on a Colab T4 GPU.

The notebook is derived from `04_qwen_multi_agent_typed_colab.ipynb`.

The experiment preserves:
- typed multi-agent architecture;
- typed protocol;
- role prompts;
- state feedback logic;
- deterministic TEC tools;
- GoldRunner;
- the same five benchmark tasks;
- the same evaluation metrics and workflow limits.

Only the model deployment path and result namespace are changed.

This is not a pure size-only ablation of Qwen3.5 because the model family changes
from Qwen3.5-4B to text-only Qwen3-8B. It is a practical model replacement
experiment intended to test whether a larger, infrastructure-stable text-only
Qwen model executes typed scientific tool workflows more reliably on Tesla T4.

Previous Qwen3.5-9B deployment attempts are excluded from model-quality
comparison because they failed at the infrastructure-validation stage:
- AWQ failed during model loading due to backend/kernel incompatibility;
- pre-quantized BNB loaded but produced corrupted text on a trivial direct prompt;
- GPTQ failed during loading/backend validation before semantic generation;
- AutoRound/vLLM loaded weights but did not reach a usable text-generation server
  in the tested T4 runtime.


## 1. Clean Colab setup

Run this in a fresh Google Colab T4 runtime. After the installation cell finishes, restart the Colab runtime once before importing torch, transformers or bitsandbytes.

This notebook uses the stable Transformers path for text-only Qwen3 and dynamic BitsAndBytes NF4 4-bit quantization. It does not install vLLM, GPTQModel, AutoAWQ, llama.cpp, or multimodal Qwen3.5-specific runtime dependencies.


In [ ]:
# Colab T4 environment setup for Qwen3-8B + dynamic BNB NF4.
import subprocess
import sys

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "transformers>=4.51.0",
        "accelerate",
        "bitsandbytes",
        "safetensors",
        "huggingface_hub",
        "nvidia-ml-py",
        "pandas",
        "pyarrow",
        "pydantic",
        "python-dateutil",
    ],
    check=True,
)


IMPORTANT: after setup, restart the Colab runtime once before running the import-check cell below.


## 2. Import check


In [ ]:
import subprocess
import sys

import bitsandbytes as bnb
import torch
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "CUDA is unavailable. Use a Colab T4 GPU runtime."

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.used,memory.free,memory.total",
    "--format=csv,noheader",
], check=True)


## 3. Clone or update repository


In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if PROJECT_DIR.exists() and (PROJECT_DIR / ".git").exists():
    tracked_changed = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "diff", "--quiet"],
        check=False,
    ).returncode != 0
    staged_changed = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "diff", "--cached", "--quiet"],
        check=False,
    ).returncode != 0
    if tracked_changed or staged_changed:
        raise RuntimeError(
            "Tracked local changes exist in the Colab checkout. Commit or stash them before updating."
        )
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists():
    raise RuntimeError(f"{PROJECT_DIR} exists but is not a git checkout.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project dir:", PROJECT_DIR)
subprocess.run(["git", "-C", str(PROJECT_DIR), "log", "-1", "--oneline"], check=True)


## 4. Project imports


In [ ]:
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

from tec_agents.agents.llm_multi_agent import initial_multi_agent_context
from tec_agents.agents.llm_multi_agent_typed import (
    LLMFullTypedMultiAgent,
    LLMTypedOrchestratorAgent,
)
from tec_agents.agents.llm_multi_agent_typed_prompts import ARCHITECTURE_NAME
from tec_agents.agents.llm_multi_agent_typed_protocol import TYPED_PROTOCOL_VERSION
from tec_agents.agents.llm_single_agent import infer_task_state
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.five_task_configs import (
    get_five_task_configs,
    build_five_task_eval_task,
    build_five_task_expected_sequence,
)
from tec_agents.eval.gold_runner import GoldRunner
from tec_agents.eval.metrics import MetricResult, compare_agent_to_gold
from tec_agents.eval.task_set import task_to_dict
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server


## 5. CONFIG


In [ ]:
# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen3-8B"
MODEL_FAMILY = "Qwen3"
MODEL_SIZE_LABEL = "8B"
MODEL_ABLATION_ID = "qwen3_8b_bnb_nf4_dynamic"

QUANTIZATION_FORMAT = "BNB_NF4_DYNAMIC_4BIT"
QUANTIZATION_SOURCE = "dynamic_quantization_of_official_checkpoint"
LOAD_IN_4BIT = True
LOAD_IN_8BIT = False

TORCH_DTYPE = "float16"
BNB_4BIT_COMPUTE_DTYPE = "float16"
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_4BIT_USE_DOUBLE_QUANT = True

ENABLE_THINKING = False

MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
DO_SAMPLE = False

# ------------------------------------------------------------
# Dataset configuration: inherited from notebook 04
# ------------------------------------------------------------

DATASET_REF = "default"
DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"
PROCESSED_DATASET_PATH = PROJECT_DIR / "data" / "processed" / DATASET_FILENAME
START = "2024-03-01"
END = "2024-04-01"

# ------------------------------------------------------------
# Architecture configuration: copied from actual notebook 04
# ------------------------------------------------------------

ARCHITECTURE_MODE = "qwen_multi_agent_typed_full_llm"
PROMPT_REVISION = "grounded_inputs_deliverables_single_block_v3"

EXPERIMENT_ID = "experiment_qwen3_8b_bnb_nf4_dynamic_typed_colab"

MAX_ORCHESTRATION_STEPS = 12
MAX_ROLE_STEPS = 8
MAX_TOOL_CALLS_PER_ROLE = 8
MAX_PARSE_RETRIES = 2
MAX_TOOL_RETRIES = 2
STATE_FEEDBACK_MODE = "typed_state"

# ------------------------------------------------------------
# Execution guard
# ------------------------------------------------------------

RUN_SEMANTIC_SMOKE_TESTS = True
RUN_ORCHESTRATOR_SMOKE_TEST = True
RUN_PILOT_TASK = True

# Full five-task batch must be manually enabled after reviewing the pilot.
RUN_ALL_TASKS = False
SELECTED_PRESET = "hightec_midlat_europe"

# ------------------------------------------------------------
# Output isolation
# ------------------------------------------------------------

OUTPUT_ROOT = PROJECT_DIR / "outputs" / "metrics"
OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_OUTPUT_PATH = OUTPUT_DIR / "qwen_multi_agent_typed_qwen3_8b_bnb_nf4_dynamic_batch_colab.json"
PER_TASK_OUTPUT_TEMPLATE = "qwen_multi_agent_typed_qwen3_8b_bnb_nf4_dynamic_{preset_id}_colab.json"
DIRECT_SMOKE_OUTPUT_PATH = OUTPUT_DIR / "qwen3_8b_bnb_nf4_dynamic_direct_smoke_tests.json"
PILOT_OUTPUT_PATH = OUTPUT_DIR / "qwen_multi_agent_typed_qwen3_8b_bnb_nf4_dynamic_pilot_hightec_midlat_europe_colab.json"

TASK_CONFIGS = get_five_task_configs(dataset_ref=DATASET_REF)
if RUN_ALL_TASKS:
    SELECTED_TASK_CONFIGS = TASK_CONFIGS
else:
    SELECTED_TASK_CONFIGS = [c for c in TASK_CONFIGS if c["preset_id"] == SELECTED_PRESET]
assert SELECTED_TASK_CONFIGS, f"Unknown SELECTED_PRESET={SELECTED_PRESET!r}"

print("Architecture:", ARCHITECTURE_MODE)
print("Typed protocol:", TYPED_PROTOCOL_VERSION)
print("Prompt revision:", PROMPT_REVISION)
print("Model:", MODEL_NAME)
print("Model ablation:", MODEL_ABLATION_ID)
print("Output dir:", OUTPUT_DIR)
print("RUN_ALL_TASKS:", RUN_ALL_TASKS)
print("Selected tasks:", [c["preset_id"] for c in SELECTED_TASK_CONFIGS])


## Planned test questions

This preview is for the human notebook reader only. The expected tool sequence is used later for evaluation and is not passed into the typed LLM agents.

In [ ]:
preview_rows = []
for config in SELECTED_TASK_CONFIGS:
    preview_rows.append({
        "preset_id": config["preset_id"],
        "task_type": config["task_type"],
        "query": config["query"],
        "expected_tool_sequence": " -> ".join(build_five_task_expected_sequence(config)),
    })
pd.DataFrame(preview_rows)

## 6. Dataset setup

In [ ]:
DATASET_PATH = PROCESSED_DATASET_PATH
assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH}"
register_dataset(dataset_ref=DATASET_REF, path=DATASET_PATH, file_format="parquet")
get_dataset_summary(DATASET_REF)

## 6. Helpers


In [ ]:
def to_jsonable(value):
    if hasattr(value, "to_dict"):
        return to_jsonable(value.to_dict())
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if hasattr(value, "item"):
        return value.item()
    return value


def metric_dict(metric_result):
    if metric_result is None:
        return {}
    return metric_result.metrics if isinstance(metric_result, MetricResult) else dict(metric_result)


def final_answer_preview(result, limit=360):
    text = getattr(result, "answer", "") or ""
    return text[:limit] + ("..." if len(text) > limit else "")


def build_verdict_checks(result, metric_result):
    metrics = metric_dict(metric_result)
    return {
        "agent_success": bool(result.success),
        "tool_sequence_match": metrics.get("tool_sequence_match"),
        "final_answer_present": bool(result.answer),
        "no_parse_errors": result.parse_error_count == 0,
        "no_stall": not result.stalled_loop_detected,
        "no_forbidden_tool_calls": result.forbidden_tool_call_count == 0,
        "no_premature_role_completion": result.premature_role_completion_count == 0,
        "no_empty_analysis_findings": result.empty_findings_done_count == 0,
    }


def overall_ok_from_checks(checks):
    required = [v for v in checks.values() if isinstance(v, bool)]
    return bool(required) and all(required)


def key_metric_summary(task_type, metrics):
    if task_type == "high_tec":
        return f"threshold_abs_error={metrics.get('threshold_abs_error')}; interval_count_error={metrics.get('interval_count_error')}"
    if task_type == "stable_intervals":
        return f"stable_interval_count_error={metrics.get('stable_interval_count_error')}"
    if task_type == "compare_regions":
        return f"mean_abs_error_avg={metrics.get('mean_abs_error_avg')}; pairwise_delta_count_match={metrics.get('pairwise_delta_count_match')}"
    if task_type == "report":
        return f"required_artifacts_present={metrics.get('required_artifacts_present')}; grounded={metrics.get('report_grounded_in_artifacts')}"
    return ""


def _torch_dtype_from_name(name):
    if name == "float16":
        return torch.float16
    if name == "bfloat16":
        return torch.bfloat16
    if name == "float32":
        return torch.float32
    raise ValueError(f"Unsupported dtype name: {name}")


def inference_metadata():
    return {
        "model_name": MODEL_NAME,
        "model_family": MODEL_FAMILY,
        "model_size_label": MODEL_SIZE_LABEL,
        "model_ablation_id": MODEL_ABLATION_ID,
        "quantization_format": QUANTIZATION_FORMAT,
        "quantization_source": QUANTIZATION_SOURCE,
        "load_in_4bit": LOAD_IN_4BIT,
        "load_in_8bit": LOAD_IN_8BIT,
        "torch_dtype": TORCH_DTYPE,
        "bnb_4bit_compute_dtype": BNB_4BIT_COMPUTE_DTYPE,
        "bnb_4bit_quant_type": BNB_4BIT_QUANT_TYPE,
        "bnb_4bit_use_double_quant": BNB_4BIT_USE_DOUBLE_QUANT,
        "enable_thinking": ENABLE_THINKING,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "do_sample": DO_SAMPLE,
        "direct_smoke_tests_passed": globals().get("DIRECT_SMOKE_TESTS_PASSED"),
        "orchestrator_smoke_passed": globals().get("ORCHESTRATOR_SMOKE_PASSED"),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "bitsandbytes_version": bnb.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "cuda_capability": list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
    }


## 7. Qwen3-8B dynamic BNB NF4 loading

This notebook intentionally does not use `LocalQwenChatModel(load_in_4bit=True)`, because the shared wrapper uses BF16 compute dtype for 4-bit quantization. This T4 branch uses a notebook-local adapter with `bnb_4bit_compute_dtype=torch.float16` and `enable_thinking=False`.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def _apply_stop_strings(text: str, stop: list[str] | None) -> str:
    if not stop:
        return text
    cut_at = None
    for item in stop:
        idx = text.find(item)
        if idx >= 0 and (cut_at is None or idx < cut_at):
            cut_at = idx
    return text if cut_at is None else text[:cut_at].strip()


class Qwen3_8BBNBChatModel:
    def __init__(self, model_name: str, max_input_tokens: int | None = None):
        self.model_name = model_name
        self.max_input_tokens = max_input_tokens
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            use_fast=True,
            trust_remote_code=True,
        )
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
            bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
            bnb_4bit_compute_dtype=torch.float16,
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        ).eval()
        if hasattr(self.model, "config"):
            self.model.config.use_cache = True

    def generate(
        self,
        messages: list[dict[str, str]],
        max_new_tokens: int = 512,
        temperature: float = 0.0,
        do_sample: bool = False,
        stop: list[str] | None = None,
    ) -> str:
        try:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        tokenized = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=self.max_input_tokens is not None,
            max_length=self.max_input_tokens,
        )
        tokenized = tokenized.to(self._input_device())
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": do_sample,
            "use_cache": True,
            "pad_token_id": self.tokenizer.eos_token_id,
        }
        if do_sample:
            generation_kwargs["temperature"] = temperature
        with torch.inference_mode():
            output_ids = self.model.generate(**tokenized, **generation_kwargs)
        input_length = tokenized["input_ids"].shape[1]
        generated_ids = output_ids[0][input_length:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        return _apply_stop_strings(text, stop)

    def _input_device(self):
        device = getattr(self.model, "device", None)
        if device is not None:
            return device
        return next(self.model.parameters()).device


model = Qwen3_8BBNBChatModel(
    model_name=MODEL_NAME,
    max_input_tokens=MAX_INPUT_TOKENS,
)
print("Model loaded:", MODEL_NAME)
print("Python class:", type(model.model).__name__)
print("Device:", next(model.model.parameters()).device)
print("4-bit:", getattr(model.model, "is_loaded_in_4bit", None))
print("Thinking disabled:", ENABLE_THINKING)
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.used,memory.free,memory.total",
    "--format=csv,noheader",
], check=True)


## 8. Direct model smoke-tests


In [ ]:
SMOKE_TESTS = {
    "semantic_ready": {
        "messages": [
            {
                "role": "system",
                "content": "Follow the user's instruction exactly. Output only the requested answer.",
            },
            {"role": "user", "content": "Reply with exactly one word: ready"},
        ],
        "max_new_tokens": 16,
    },
    "simple_math": {
        "messages": [
            {"role": "system", "content": "Output only the requested answer."},
            {"role": "user", "content": "What is 7 multiplied by 8? Reply with only the number."},
        ],
        "max_new_tokens": 16,
    },
    "typed_xml": {
        "messages": [
            {"role": "system", "content": "Return exactly one XML block and nothing else."},
            {
                "role": "user",
                "content": (
                    "Return exactly this text:\n"
                    "<role_action>\n"
                    '{"role":"DataAgent","deliverables_to_produce":["series_id"]}\n'
                    "</role_action>"
                ),
            },
        ],
        "max_new_tokens": 128,
    },
    "tool_json": {
        "messages": [
            {"role": "system", "content": "Return exactly one JSON object and nothing else."},
            {
                "role": "user",
                "content": (
                    "Return exactly this JSON object:\n"
                    '{"name":"tec_get_timeseries",'
                    '"arguments":{"region":"midlat_europe"}}'
                ),
            },
        ],
        "max_new_tokens": 128,
    },
}

outputs = {}
timings = {}
for name, item in SMOKE_TESTS.items():
    started = time.time()
    output = model.generate(
        item["messages"],
        max_new_tokens=item["max_new_tokens"],
        temperature=0.0,
        do_sample=False,
    )
    timings[name] = time.time() - started
    outputs[name] = output
    print()
    print("===", name, "===")
    print("elapsed_sec:", round(timings[name], 3))
    print("raw:", repr(output))

SEMANTIC_SMOKE_PASSED = outputs["semantic_ready"].strip().lower() == "ready"
MATH_SMOKE_PASSED = outputs["simple_math"].strip() == "56"
XML_SMOKE_PASSED = all(
    fragment in outputs["typed_xml"]
    for fragment in ["<role_action>", "</role_action>", "DataAgent", "series_id"]
)
TOOL_JSON_SMOKE_PASSED = all(
    fragment in outputs["tool_json"]
    for fragment in ["tec_get_timeseries", "arguments", "midlat_europe"]
)

DIRECT_SMOKE_TESTS_PASSED = all([
    SEMANTIC_SMOKE_PASSED,
    MATH_SMOKE_PASSED,
    XML_SMOKE_PASSED,
    TOOL_JSON_SMOKE_PASSED,
])

smoke_payload = {
    "model_name": MODEL_NAME,
    "model_ablation_id": MODEL_ABLATION_ID,
    "quantization_format": QUANTIZATION_FORMAT,
    "quantization_source": QUANTIZATION_SOURCE,
    "torch_dtype": TORCH_DTYPE,
    "bnb_4bit_compute_dtype": BNB_4BIT_COMPUTE_DTYPE,
    "enable_thinking": ENABLE_THINKING,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "outputs": outputs,
    "timings_sec": timings,
    "verdict": {
        "semantic_ready": SEMANTIC_SMOKE_PASSED,
        "simple_math": MATH_SMOKE_PASSED,
        "typed_xml": XML_SMOKE_PASSED,
        "tool_json": TOOL_JSON_SMOKE_PASSED,
    },
    "direct_smoke_tests_passed": DIRECT_SMOKE_TESTS_PASSED,
    "inference_metadata": inference_metadata(),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
with DIRECT_SMOKE_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(smoke_payload), f, ensure_ascii=False, indent=2)
print("Saved smoke results:", DIRECT_SMOKE_OUTPUT_PATH)

assert DIRECT_SMOKE_TESTS_PASSED, (
    "Direct model smoke-tests failed. Do not run the typed multi-agent benchmark."
)


## 9. One-turn Orchestrator smoke-test


In [ ]:
if RUN_ORCHESTRATOR_SMOKE_TEST:
    hightec_config = next(c for c in TASK_CONFIGS if c["preset_id"] == "hightec_midlat_europe")
    hightec_query = hightec_config["query"]
    parsed_task = infer_task_state(hightec_query)
    context = initial_multi_agent_context(user_query=hightec_query, parsed_task=parsed_task)
    orchestrator = LLMTypedOrchestratorAgent(
        model,
        temperature=TEMPERATURE,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    role_action, orchestrator_diagnostics = orchestrator.decide(
        user_query=hightec_query,
        context=context,
        max_parse_retries=0,
    )
    raw_orchestrator_output = (orchestrator_diagnostics.get("raw_model_outputs") or [""])[-1]
    cleaned_orchestrator_output = (orchestrator_diagnostics.get("cleaned_model_outputs") or [""])[-1]
    print("Raw Orchestrator output")
    print(raw_orchestrator_output)
    print("Cleaned Orchestrator output")
    print(cleaned_orchestrator_output)
    print("Parsed action:", role_action)
    print("Selected role:", getattr(role_action, "role", None))
    print("Assignment:", getattr(role_action, "assignment", None))
    print("Diagnostics:", orchestrator_diagnostics)
    ORCHESTRATOR_SMOKE_PASSED = (
        role_action is not None
        and role_action.action == "call_role"
        and role_action.assignment is not None
        and orchestrator_diagnostics.get("parse_error_count", 0) == 0
    )
else:
    ORCHESTRATOR_SMOKE_PASSED = True

assert ORCHESTRATOR_SMOKE_PASSED, "Orchestrator smoke-test has not passed."


## 10. Pilot task: hightec_midlat_europe


In [ ]:
def run_one_task(task_config, *, run_id, output_path):
    query = task_config["query"]
    expected_sequence = build_five_task_expected_sequence(task_config)
    eval_task = build_five_task_eval_task(task_config)
    server = build_local_mcp_server(run_id=run_id)
    client = LocalMCPClient(server)
    agent = LLMFullTypedMultiAgent(
        model=model,
        client=client,
        max_orchestration_steps=MAX_ORCHESTRATION_STEPS,
        max_role_steps=MAX_ROLE_STEPS,
        max_tool_calls_per_role=MAX_TOOL_CALLS_PER_ROLE,
        max_parse_retries=MAX_PARSE_RETRIES,
        max_tool_retries=MAX_TOOL_RETRIES,
        temperature=TEMPERATURE,
        state_feedback_mode=STATE_FEEDBACK_MODE,
    )
    result = agent.run(query)
    result_dict = result.to_dict()

    gold_runner = GoldRunner()
    gold = gold_runner.run(eval_task)
    metric_result = None
    if gold.status == "success" and gold.result is not None:
        agent_metric_payload = dict(result.tool_results)
        agent_metric_payload.update({
            "parse_error_count": result.parse_error_count,
            "invalid_json_count": result.invalid_json_count,
            "invalid_role_protocol_count": result.invalid_role_protocol_count,
            "forbidden_tool_call_count": result.forbidden_tool_call_count,
            "repeated_tool_call_count": result.repeated_tool_call_count,
            "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
            "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
            "stalled_loop_detected": result.stalled_loop_detected,
        })
        metric_result = compare_agent_to_gold(
            task_id=eval_task.task_id,
            task_type=eval_task.task_type,
            agent_result=agent_metric_payload,
            gold_result=gold.result,
            agent_trace=result.trace,
            task=task_to_dict(eval_task),
            parsed_task=result.parsed_task,
            orchestration_steps=result.orchestration_steps,
        )

    metrics = metric_dict(metric_result)
    verdict_checks = build_verdict_checks(result, metric_result)
    record = {
        "selected_preset": task_config["preset_id"],
        "task_config": task_config,
        "query": query,
        "expected_tool_sequence": expected_sequence,
        "actual_tool_sequence": result.actual_tool_sequence,
        "architecture": ARCHITECTURE_MODE,
        "model_name": MODEL_NAME,
        "model_ablation_id": MODEL_ABLATION_ID,
        "inference_metadata": inference_metadata(),
        "typed_protocol_version": TYPED_PROTOCOL_VERSION,
        "prompt_revision": PROMPT_REVISION,
        "agent_result": result_dict,
        "gold_status": gold.status,
        "gold_error": gold.error,
        "gold_result": gold.result,
        "metrics": metrics,
        "verdict_checks": verdict_checks,
        "overall_ok": overall_ok_from_checks(verdict_checks),
        "success": bool(result.success),
        "final_answer_preview": final_answer_preview(result),
        "role_agent_order": result.role_agent_order,
        "orchestrator_decisions": result.orchestrator_decisions,
        "role_actions": result.role_actions,
        "role_assignments": result.role_assignments,
        "role_outputs": result.role_outputs,
        "tool_observations": result.tool_observations,
        "available_artifacts_snapshots": result.available_artifacts_snapshots,
        "invalid_assignment_count": result.invalid_assignment_count,
        "invalid_role_action_count": result.invalid_role_action_count,
        "invalid_role_response_count": result.invalid_role_response_count,
        "invalid_final_answer_count": result.invalid_final_answer_count,
        "invalid_tool_name_count": result.invalid_tool_name_count,
        "forbidden_tool_call_count": result.forbidden_tool_call_count,
        "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
        "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
        "premature_role_completion_count": result.premature_role_completion_count,
        "empty_findings_done_count": result.empty_findings_done_count,
        "repeated_equivalent_role_assignment_count": result.repeated_equivalent_role_assignment_count,
        "tool_error_count": result.tool_error_count,
        "tool_schema_validation_error_count": result.tool_schema_validation_error_count,
        "invalid_role_protocol_count": result.invalid_role_protocol_count,
        "parse_error_count": result.parse_error_count,
        "invalid_json_count": result.invalid_json_count,
        "repeated_tool_call_count": result.repeated_tool_call_count,
        "stalled_loop_detected": result.stalled_loop_detected,
        "repair_attempt_count": result.repair_attempt_count,
        "retry_count": result.retry_count,
        "recovery_attempt_count": result.recovery_attempt_count,
        "recovery_success_count": result.recovery_success_count,
        "recovery_failure_count": result.recovery_failure_count,
        "key_metric_summary": key_metric_summary(task_config["task_type"], metrics),
        "missing_agent_terminal_artifact": (metrics or {}).get("missing_agent_terminal_artifact"),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(record), f, ensure_ascii=False, indent=2)
    return record


if RUN_PILOT_TASK:
    assert DIRECT_SMOKE_TESTS_PASSED, "Direct smoke-tests have not passed."
    assert ORCHESTRATOR_SMOKE_PASSED, "Orchestrator smoke-test has not passed."
    pilot_config = next(c for c in TASK_CONFIGS if c["preset_id"] == "hightec_midlat_europe")
    pilot_record = run_one_task(
        pilot_config,
        run_id="qwen_multi_agent_typed_qwen3_8b_bnb_nf4_dynamic_pilot_hightec_colab",
        output_path=PILOT_OUTPUT_PATH,
    )
    print("pilot success:", pilot_record["success"])
    print("pilot overall_ok:", pilot_record["overall_ok"])
    print("pilot actual_tool_sequence:", pilot_record["actual_tool_sequence"])
    print("pilot role_agent_order:", pilot_record["role_agent_order"])
    print("pilot tool_call_count:", len(pilot_record["actual_tool_sequence"]))
    print("pilot tool_error_count:", pilot_record["tool_error_count"])
    print("pilot parse_error_count:", pilot_record["parse_error_count"])
    print("pilot invalid_artifact_handle_count:", pilot_record["invalid_artifact_handle_count"])
    print("pilot multiple_protocol_blocks_in_single_output_count:", pilot_record["multiple_protocol_blocks_in_single_output_count"])
    print("pilot repeated_tool_call_count:", pilot_record["repeated_tool_call_count"])
    print("pilot stalled_loop_detected:", pilot_record["stalled_loop_detected"])
    print("pilot final_answer_preview:", pilot_record["final_answer_preview"])
    print("Saved pilot:", PILOT_OUTPUT_PATH)


## Full five-task benchmark

Do not execute the full five-task benchmark until the pilot result has been reviewed. To start the complete batch, set `RUN_ALL_TASKS = True` in CONFIG, rerun the task-selection cell, and then execute the batch cell below.


In [ ]:
assert DIRECT_SMOKE_TESTS_PASSED, "Direct model smoke-tests have not passed."
if "ORCHESTRATOR_SMOKE_PASSED" in globals():
    assert ORCHESTRATOR_SMOKE_PASSED, "Orchestrator smoke-test has not passed."

if not RUN_ALL_TASKS:
    print(
        "Full batch is disabled. "
        "Review the pilot result, then set RUN_ALL_TASKS = True "
        "in CONFIG and rerun the selection and batch cells."
    )
    all_results = []
else:
    all_results = []
    gold_runner = GoldRunner()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for task_config in SELECTED_TASK_CONFIGS:
        preset_id = task_config["preset_id"]
        query = task_config["query"]
        expected_sequence = build_five_task_expected_sequence(task_config)
        eval_task = build_five_task_eval_task(task_config)

        print()
        print("===", preset_id, "===")
        server = build_local_mcp_server(run_id=f"qwen_multi_agent_typed_qwen3_8b_bnb_nf4_dynamic_{preset_id}_colab")
        client = LocalMCPClient(server)
        agent = LLMFullTypedMultiAgent(
            model=model,
            client=client,
            max_orchestration_steps=MAX_ORCHESTRATION_STEPS,
            max_role_steps=MAX_ROLE_STEPS,
            max_tool_calls_per_role=MAX_TOOL_CALLS_PER_ROLE,
            max_parse_retries=MAX_PARSE_RETRIES,
            max_tool_retries=MAX_TOOL_RETRIES,
            temperature=TEMPERATURE,
            state_feedback_mode=STATE_FEEDBACK_MODE,
        )

        result = agent.run(query)
        result_dict = result.to_dict()

        gold = gold_runner.run(eval_task)
        metric_result = None
        gold_status = gold.status
        gold_error = gold.error
        gold_result = gold.result
        if gold.status == "success" and gold.result is not None:
            agent_metric_payload = dict(result.tool_results)
            agent_metric_payload.update({
                "parse_error_count": result.parse_error_count,
                "invalid_json_count": result.invalid_json_count,
                "invalid_role_protocol_count": result.invalid_role_protocol_count,
                "forbidden_tool_call_count": result.forbidden_tool_call_count,
                "repeated_tool_call_count": result.repeated_tool_call_count,
                "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
                "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
                "stalled_loop_detected": result.stalled_loop_detected,
            })
            metric_result = compare_agent_to_gold(
                task_id=eval_task.task_id,
                task_type=eval_task.task_type,
                agent_result=agent_metric_payload,
                gold_result=gold.result,
                agent_trace=result.trace,
                task=task_to_dict(eval_task),
                parsed_task=result.parsed_task,
                orchestration_steps=result.orchestration_steps,
            )

        metrics = metric_dict(metric_result)
        verdict_checks = build_verdict_checks(result, metric_result)
        overall_ok = overall_ok_from_checks(verdict_checks)

        record = {
            "selected_preset": preset_id,
            "task_config": task_config,
            "query": query,
            "expected_tool_sequence": expected_sequence,
            "actual_tool_sequence": result.actual_tool_sequence,
            "architecture": ARCHITECTURE_MODE,
            "model_name": MODEL_NAME,
            "model_ablation_id": MODEL_ABLATION_ID,
            "inference_metadata": inference_metadata(),
            "typed_protocol_version": TYPED_PROTOCOL_VERSION,
            "prompt_revision": PROMPT_REVISION,
            "agent_result": result_dict,
            "gold_status": gold_status,
            "gold_error": gold_error,
            "gold_result": gold_result,
            "metrics": metrics,
            "verdict_checks": verdict_checks,
            "overall_ok": overall_ok,
            "success": bool(result.success),
            "final_answer_preview": final_answer_preview(result),
            "role_agent_order": result.role_agent_order,
            "orchestrator_decisions": result.orchestrator_decisions,
            "role_actions": result.role_actions,
            "role_assignments": result.role_assignments,
            "role_outputs": result.role_outputs,
            "tool_observations": result.tool_observations,
            "available_artifacts_snapshots": result.available_artifacts_snapshots,
            "invalid_assignment_count": result.invalid_assignment_count,
            "invalid_role_action_count": result.invalid_role_action_count,
            "invalid_role_response_count": result.invalid_role_response_count,
            "invalid_final_answer_count": result.invalid_final_answer_count,
            "invalid_tool_name_count": result.invalid_tool_name_count,
            "forbidden_tool_call_count": result.forbidden_tool_call_count,
            "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
            "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
            "premature_role_completion_count": result.premature_role_completion_count,
            "empty_findings_done_count": result.empty_findings_done_count,
            "repeated_equivalent_role_assignment_count": result.repeated_equivalent_role_assignment_count,
            "tool_error_count": result.tool_error_count,
            "tool_schema_validation_error_count": result.tool_schema_validation_error_count,
            "invalid_role_protocol_count": result.invalid_role_protocol_count,
            "parse_error_count": result.parse_error_count,
            "invalid_json_count": result.invalid_json_count,
            "repeated_tool_call_count": result.repeated_tool_call_count,
            "stalled_loop_detected": result.stalled_loop_detected,
            "repair_attempt_count": result.repair_attempt_count,
            "retry_count": result.retry_count,
            "recovery_attempt_count": result.recovery_attempt_count,
            "recovery_success_count": result.recovery_success_count,
            "recovery_failure_count": result.recovery_failure_count,
            "key_metric_summary": key_metric_summary(task_config["task_type"], metrics),
            "missing_agent_terminal_artifact": (metrics or {}).get("missing_agent_terminal_artifact"),
        }

        per_task_path = OUTPUT_DIR / PER_TASK_OUTPUT_TEMPLATE.format(preset_id=preset_id)
        with per_task_path.open("w", encoding="utf-8") as f:
            json.dump(to_jsonable(record), f, ensure_ascii=False, indent=2)
        print("Saved:", per_task_path)
        print("success:", record["success"], "overall_ok:", overall_ok, "tools:", result.actual_tool_sequence)
        all_results.append(record)


## Summary table and save batch


In [ ]:
if not RUN_ALL_TASKS:
    print("Full batch has not been run yet; summary will be available after RUN_ALL_TASKS=True.")
else:
    def rate(values):
        values = [v for v in values if isinstance(v, bool)]
        return None if not values else sum(values) / len(values)

    summary = {
        "n_tasks": len(all_results),
        "n_success": sum(1 for r in all_results if r["success"]),
        "n_failed": sum(1 for r in all_results if not r["success"]),
        "success_rate": rate([r["success"] for r in all_results]),
        "n_overall_ok": sum(1 for r in all_results if r["overall_ok"]),
        "overall_ok_rate": rate([r["overall_ok"] for r in all_results]),
        "agent_success_rate": rate([r["success"] for r in all_results]),
        "tool_sequence_match_rate": rate([(r.get("metrics") or {}).get("tool_sequence_match") for r in all_results]),
        "role_agent_order_match_rate": rate([(r.get("metrics") or {}).get("role_agent_order_match") for r in all_results]),
        "artifact_flow_valid_rate": rate([(r.get("metrics") or {}).get("artifact_flow_valid") for r in all_results]),
        "required_role_agents_called_rate": rate([(r.get("metrics") or {}).get("required_role_agents_called") for r in all_results]),
        "avg_tool_call_count": sum(len(r["actual_tool_sequence"]) for r in all_results) / max(1, len(all_results)),
        "avg_tool_error_count": sum(r["tool_error_count"] for r in all_results) / max(1, len(all_results)),
        "avg_parse_error_count": sum(r["parse_error_count"] for r in all_results) / max(1, len(all_results)),
        "avg_invalid_assignment_count": sum(r["invalid_assignment_count"] for r in all_results) / max(1, len(all_results)),
        "avg_invalid_role_response_count": sum(r["invalid_role_response_count"] for r in all_results) / max(1, len(all_results)),
        "avg_forbidden_tool_call_count": sum(r["forbidden_tool_call_count"] for r in all_results) / max(1, len(all_results)),
        "avg_multiple_protocol_blocks_in_single_output_count": sum(r["multiple_protocol_blocks_in_single_output_count"] for r in all_results) / max(1, len(all_results)),
        "avg_invalid_artifact_handle_count": sum(r["invalid_artifact_handle_count"] for r in all_results) / max(1, len(all_results)),
        "avg_premature_role_completion_count": sum(r["premature_role_completion_count"] for r in all_results) / max(1, len(all_results)),
        "avg_empty_findings_done_count": sum(r["empty_findings_done_count"] for r in all_results) / max(1, len(all_results)),
        "avg_repeated_equivalent_role_assignment_count": sum(r["repeated_equivalent_role_assignment_count"] for r in all_results) / max(1, len(all_results)),
        "avg_tool_schema_validation_error_count": sum(r["tool_schema_validation_error_count"] for r in all_results) / max(1, len(all_results)),
        "stalled_loop_rate": rate([r["stalled_loop_detected"] for r in all_results]),
        "legacy_report_tool_used_rate": rate([("tec_" + "build_report") in r["actual_tool_sequence"] for r in all_results]),
    }

    payload = {
        "experiment_id": EXPERIMENT_ID,
        "architecture": ARCHITECTURE_MODE,
        "model_name": MODEL_NAME,
        "model_ablation_id": MODEL_ABLATION_ID,
        "inference_metadata": inference_metadata(),
        "dataset_ref": DATASET_REF,
        "dataset_path": str(DATASET_PATH),
        "model_config": {
            "model_name": MODEL_NAME,
            "model_family": MODEL_FAMILY,
            "model_size_label": MODEL_SIZE_LABEL,
            "model_ablation_id": MODEL_ABLATION_ID,
            "quantization_format": QUANTIZATION_FORMAT,
            "quantization_source": QUANTIZATION_SOURCE,
            "load_in_4bit": LOAD_IN_4BIT,
            "load_in_8bit": LOAD_IN_8BIT,
            "torch_dtype": TORCH_DTYPE,
            "bnb_4bit_compute_dtype": BNB_4BIT_COMPUTE_DTYPE,
            "bnb_4bit_quant_type": BNB_4BIT_QUANT_TYPE,
            "bnb_4bit_use_double_quant": BNB_4BIT_USE_DOUBLE_QUANT,
            "enable_thinking": ENABLE_THINKING,
            "max_input_tokens": MAX_INPUT_TOKENS,
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "do_sample": DO_SAMPLE,
        },
        "agent_config": {
            "max_orchestration_steps": MAX_ORCHESTRATION_STEPS,
            "max_role_steps": MAX_ROLE_STEPS,
            "max_tool_calls_per_role": MAX_TOOL_CALLS_PER_ROLE,
            "max_parse_retries": MAX_PARSE_RETRIES,
            "max_tool_retries": MAX_TOOL_RETRIES,
            "state_feedback_mode": STATE_FEEDBACK_MODE,
        },
        "typed_protocol_version": TYPED_PROTOCOL_VERSION,
        "prompt_revision": PROMPT_REVISION,
        "summary": summary,
        "results": all_results,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    with BATCH_OUTPUT_PATH.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(payload), f, ensure_ascii=False, indent=2)

    summary_rows = []
    for r in all_results:
        summary_rows.append({
            "preset_id": r["selected_preset"],
            "task_type": r["task_config"]["task_type"],
            "agent_success": r["success"],
            "overall_ok": r["overall_ok"],
            "tool_sequence_match": (r.get("metrics") or {}).get("tool_sequence_match"),
            "actual_tool_sequence": " -> ".join(r["actual_tool_sequence"]),
            "role_agent_order": " -> ".join(r["role_agent_order"]),
            "tool_call_count": len(r["actual_tool_sequence"]),
            "tool_error_count": r["tool_error_count"],
            "final_answer_present": bool((r["agent_result"] or {}).get("answer")),
            "role_agent_order_match": (r.get("metrics") or {}).get("role_agent_order_match"),
            "required_role_agents_called": (r.get("metrics") or {}).get("required_role_agents_called"),
            "artifact_flow_valid": (r.get("metrics") or {}).get("artifact_flow_valid"),
            "invalid_artifact_handle_count": r["invalid_artifact_handle_count"],
            "multiple_protocol_blocks_in_single_output_count": r["multiple_protocol_blocks_in_single_output_count"],
            "premature_role_completion_count": r["premature_role_completion_count"],
            "empty_findings_done_count": r["empty_findings_done_count"],
            "repeated_equivalent_role_assignment_count": r["repeated_equivalent_role_assignment_count"],
            "tool_schema_validation_error_count": r["tool_schema_validation_error_count"],
            "parse_error_count": r["parse_error_count"],
            "repeated_tool_call_count": r["repeated_tool_call_count"],
            "forbidden_tool_call_count": r["forbidden_tool_call_count"],
            "stalled_loop_detected": r["stalled_loop_detected"],
            "missing_agent_terminal_artifact": r.get("missing_agent_terminal_artifact"),
            "key_metric_summary": r["key_metric_summary"],
            "answer_preview": r["final_answer_preview"],
        })
    print("Saved batch:", BATCH_OUTPUT_PATH)
    pd.DataFrame(summary_rows)


## Optional comparison


In [ ]:
def load_optional(path):
    if not path.exists():
        print("Missing:", path)
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


comparison_sources = {
    "qwen_single": load_optional(OUTPUT_ROOT / "qwen_single_agent_batch_colab.json"),
    "rule_multi": load_optional(OUTPUT_ROOT / "multi_agent_rule_based_five_task_batch.json"),
    "qwen_multi_old": load_optional(OUTPUT_ROOT / "qwen_multi_agent_batch_colab.json"),
    "typed_v1_minimal_role_response": load_optional(OUTPUT_ROOT / "qwen_multi_agent_typed_batch_colab.json"),
    "typed_v2_role_boundaries_tool_schemas": load_optional(OUTPUT_ROOT / "experiment_3" / "qwen_multi_agent_typed_v2_batch_colab.json"),
    "typed_v3_grounded_inputs_deliverables_single_block": load_optional(OUTPUT_ROOT / "experiment_4" / "qwen_multi_agent_typed_v3_batch_colab.json"),
    "qwen3_8b_bnb_nf4_dynamic": load_optional(BATCH_OUTPUT_PATH),
}

source_rows = []
for source_name, payload in comparison_sources.items():
    summary = (payload or {}).get("summary") or {}
    source_rows.append({
        "source": source_name,
        "present": payload is not None,
        "architecture": (payload or {}).get("architecture"),
        "model_name": (payload or {}).get("model_name"),
        "success_rate": summary.get("success_rate"),
        "overall_ok_rate": summary.get("overall_ok_rate"),
        "tool_sequence_match_rate": summary.get("tool_sequence_match_rate"),
        "stalled_loop_rate": summary.get("stalled_loop_rate"),
    })
pd.DataFrame(source_rows)
